# 25 · Ενημέρωση του Space με τοπικό lexicon ως primary

Έλεγχος με τα πραγματικά, καρφιτσωμένα DistilBERT και Qwen βάρη → ενημέρωση του υπάρχοντος `krimits/hotel-triage-demo` από τον λογαριασμό σου → έλεγχος της φορτωμένης έκδοσης και της τοπικής πολιτικής.

Το lexicon είναι κύρια πηγή ενδείξεων topics. Άγνωστα topics ζητούν άνθρωπο· το Qwen εξακολουθεί να εξετάζει ολόκληρη την κριτική για τεκμήρια και προτάσεις. Δεν γίνεται νέα εκπαίδευση, αξιολόγηση ακρίβειας ή επιλογή μοντέλου. Το Space παραμένει πειραματικό.

Χρειάζεται **νέα συνεδρία Colab με GPU** και Colab Secret **`HF_TOKEN`** με write access στο συγκεκριμένο Space. Δεν διαβάζονται κλειδιά Jev/OpenRouter. Τα λειτουργικά παραδείγματα είναι επινοημένα και οι γνωστές development περιπτώσεις δεν είναι κρατημένο test. Το CPU/GPU compute και το hosting εξακολουθούν να έχουν κόστος/όρια.

Το notebook ανεβάζει μόνο το allowlisted πακέτο κώδικα και manifest. Δεν ανεβάζει CSV, ανθρώπινες ετικέτες ή πραγματικές κριτικές. Το αρχικό notebook 19 παραμένει στην παλιά έκδοσή του.

### 1. Ρυθμίσεις

`DRY_RUN=False` εκτελεί την ήδη εγκεκριμένη ενημέρωση από τον λογαριασμό σου. Με `True` γίνονται οι ίδιοι τοπικοί έλεγχοι, χωρίς Hub write και χωρίς ανάγνωση token. `RESUME=True` αφορά μόνο το υπάρχον πειραματικό Space.

Η ενημέρωση θέτει **μόνο** τις δημόσιες Variables `REVIEWNLP_TRIAGE_LEXICON_FALLBACK=1` και `REVIEWNLP_JEV_ENABLED=0`. Τα υπάρχοντα Secrets και hardware διατηρούνται. Ο Jev μπορεί αργότερα να ενεργοποιηθεί προαιρετικά με διαθέσιμη πρόσβαση και το checkbox· δεν δοκιμάζεται εδώ.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

SOURCE_COMMIT = "02d6e8c7b7297bd87b4060b8f9f504421e60244c"
SPACE_ID = "krimits/hotel-triage-demo"
DRY_RUN = False
RESUME = True
SECRET_NAME = "HF_TOKEN"
REPO_DIR = Path("/content/hotel-triage-lexicon-update")
ENV_DIR = Path("/content/hotel-triage-lexicon-update-env")
RUN_NAME = "triage_lexicon_update_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = REPO_DIR / "runs" / RUN_NAME
SETUP_LOG = REPO_DIR.parent / (RUN_NAME + "_setup_execution.txt")
PACKAGE_DIR = RUN_DIR / "package"
SMOKE_REPORT = RUN_DIR / "smoke.json"
PUBLICATION_REPORT = RUN_DIR / "published.json"
SOURCE_HASHES = {'scripts/prepare_triage_space.py': 'd001b7de80fbd46300894025d77f4b1e1966934f9882d5767f8cf4124e145905', 'scripts/check_triage_space.py': '55e439dbf97fa851bbadbd9cda626c83c32437e6542352482a39e0d68c73ca63', 'scripts/publish_triage_space.py': '34e187019a5da6371c8f20c3daab5f855be477fb55c361bf69dfd7ab0f4ddc6e', 'src/reviewnlp/__init__.py': '4158879c4ff4a682da05e0287803466930d6716171670ef71d028e3777a95371', 'src/reviewnlp/absa/__init__.py': '213c2905b3cadfac06bc93ee7e509dae5dfafb8ca59c3abcca4383d1430e6911', 'src/reviewnlp/absa/aspects.py': '2e48653dc4433772a30f55d74d92e373a1d3dff79dfc2c4f80d6bbd3431ee3bd', 'src/reviewnlp/absa/extract.py': '5c6b10b8f4573b0fd3b34a3fd17062c4622c44160f6194c803f47a3cda6edcc6', 'src/reviewnlp/triage/__init__.py': 'e2f3cc51007be20106dceedbb4939778fc13ee00964ab19d87f3f496991326cb', 'src/reviewnlp/triage/schemas.py': 'e87e0f0a011d846b8a2f2e1503857958b967fb2bb1cc9e3bbae9dd0c01abcf8b', 'src/reviewnlp/triage/questions.py': '5e5e88ba8f12b7e93f11b68cdd24d23ae8b3ee2e1ba77e8e6836af849424bdb8', 'src/reviewnlp/triage/jev_client.py': 'e4cec68e57edd43338e4b7969e79078668dbc4bcec4c7e2f4f169c5c5388894d', 'src/reviewnlp/triage/routing.py': '4dc6983be79195cfef1eb84b73e8c7c2ae805a88d9b8544390eb39f3a490387e', 'src/reviewnlp/triage/qwen_generator.py': 'e6743eaa68bd9b4cb8596b72681c8fb8c43945abe5ba3a9f97812f1c940d839d', 'src/reviewnlp/triage/staged_generator.py': '5393fb6891e72545300be4926287c131468163b9fc3ade3e951fc60e77a51517', 'src/reviewnlp/triage/evidence_generator.py': '959f78e85963385e30ec275451ee2e5f92eef2cbe39dc67b355678a939b61bc3', 'src/reviewnlp/triage/generator_experiment.py': '877b1e323e6a80d1103694044e5fe80da4b7d80cace349935cd1df00b00f78de', 'src/reviewnlp/triage/pipeline.py': 'c2e2b894725922bfe5bcf4f486fc65ebd9b8e503ba4b78146fc14a71b8bd4377', 'src/reviewnlp/triage/demo_service.py': '9d73dbb59cb16a7556e78204b52677848123927d7d7d74d8aa06c4cf96156e5b', 'src/reviewnlp/triage/space_runtime.py': '2ee4b89814d129a2d6f477e0ade86f199a80bed0c217e0b77347b141ece0f28f', 'src/reviewnlp/triage/span_evidence_generator.py': '42ab41e44d599bcf817e5b561a06a3eefbe207221deb61cbfd09cc401eb4057a', 'src/reviewnlp/triage/lexicon.py': '3e55150713a40fa7e39bf078d55b07ca293e56c503ccbb7b000577d07bc29897', 'spaces/hotel-triage-demo/app.py': '8ffbb9603bfb88bfb5aaaf6533811e99e75bd9563c6706a03a16e0dd7ec39887', 'spaces/hotel-triage-demo/README.md': '7901acd0ed6e8c81aec9e3a26bfa437dc5a8559973e645cf3cf42fcd35d3a712', 'spaces/hotel-triage-demo/requirements.txt': '13bd077a66c97ad0303388b9aafc36f39fa5156dbde56cc35a44361865e5cf64'}
EXPECTED_LEXICON_VERSION = 'complaint-lexicon-v1'
EXPECTED_LEXICON_SHA256 = 'e8311382893a280ffec11e87c0c922af20ffc77e141841a691b2e058a106445c'
SETUP_READY = PACKAGE_READY = SMOKE_READY = LIVE_READY = False
print("Τοπικό lexicon primary, source:", SOURCE_COMMIT, "· Jev off · ποιότητα μη επικυρωμένη.")


## 2. Απομονωμένο περιβάλλον και SHA-256

Νέο venv, χωρίς τα προεγκατεστημένα Gradio/diffusers του Colab. Οι εκδόσεις είναι του πακέτου Space.
Η πρώτη εγκατάσταση GPU torch και η λήψη Qwen 1.5B χρειάζονται αρκετά λεπτά.

In [ ]:
SETUP_READY = PACKAGE_READY = SMOKE_READY = LIVE_READY = False
import hashlib
import json
import os
import subprocess
import sys

base_env = {name: value for name, value in os.environ.items()
            if name not in {SECRET_NAME, "OPENROUTER_API_KEY", "TYPESAFE_API_KEY", "PYTHONPATH", "SPACE_ID", "SPACES_ZERO_GPU"}}
base_env["REVIEWNLP_JEV_ENABLED"] = "0"
base_env["REVIEWNLP_TRIAGE_LEXICON_FALLBACK"] = "1"
base_env["HF_HUB_DISABLE_TELEMETRY"] = "1"
base_env["GRADIO_ANALYTICS_ENABLED"] = "False"
base_env["HF_HUB_DISABLE_XET"] = "1"
# Colab's inline backend is unavailable inside the isolated subprocess environment.
base_env["MPLBACKEND"] = "Agg"

SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)

def run_command(arguments, *, cwd=None, capture=False):
    print("Εκτέλεση:", Path(arguments[0]).name, " ".join(arguments[1:4]))
    result = subprocess.run(arguments, cwd=cwd, env=base_env, check=False, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    with SETUP_LOG.open("a", encoding="utf-8") as log:
        log.write("Command: " + " ".join(arguments) + "\n")
        log.write(result.stdout or "")
        log.write(result.stderr or "")
        log.write("\nExit: " + str(result.returncode) + "\n")
    if result.returncode:
        raise RuntimeError("Το setup σταμάτησε· κατέβασε το ZIP με setup_execution.txt από το τελευταίο κελί.")
    return result.stdout.strip() if capture else None

if not REPO_DIR.exists():
    run_command(["git", "clone", "--quiet", "https://github.com/krimits/hotel-review-nlp.git", str(REPO_DIR)])
    run_command(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], cwd=REPO_DIR)
if run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture=True) != SOURCE_COMMIT:
    raise RuntimeError("Άλλο checkout. Χρησιμοποίησε νέα συνεδρία· δεν γίνεται αυτόματο reset.")
if run_command(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, capture=True):
    raise RuntimeError("Υπάρχουν αλλαγές στο checkout.")
for name, expected in SOURCE_HASHES.items():
    if hashlib.sha256((REPO_DIR / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Διαφέρει το αρχείο: " + name)
if not ENV_DIR.exists():
    run_command([sys.executable, "-m", "venv", str(ENV_DIR)])
PYTHON = str(ENV_DIR / "bin/python")
run_command([PYTHON, "-m", "pip", "install", "--quiet", "-r",
             str(REPO_DIR / "spaces/hotel-triage-demo/requirements.txt"), "spaces==0.51.3"])
run_command([PYTHON, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO_DIR)])
run_command([PYTHON, "-m", "pip", "check"])
probe = json.loads(run_command([PYTHON, "-c",
    "import torch,json; print(json.dumps({'cuda':torch.cuda.is_available(),'torch':torch.__version__}))"], capture=True))
if not probe["cuda"]:
    raise RuntimeError("Επίλεξε GPU στο Colab πριν από το smoke test.")
RUN_DIR.mkdir(parents=True, exist_ok=False)
SETUP_READY = True
print("Checkout, hashes, απομονωμένες εξαρτήσεις και GPU: OK.", probe)


## 3. Πακέτο μόνο με κώδικα

Ο builder δεν γράφει στο Hub. Το manifest παγώνει prompts, μοντέλα, κατηγορίες, routing και όρια.
Δεν περιλαμβάνει ετικέτες, κλειδί αξιολόγησης, reviews, logits, run logs ή API keys.

In [ ]:
PACKAGE_READY = SMOKE_READY = LIVE_READY = False
if not SETUP_READY:
    raise RuntimeError("Ολοκλήρωσε πρώτα το setup.")
run_command([PYTHON, "scripts/prepare_triage_space.py", "--output", str(PACKAGE_DIR)], cwd=REPO_DIR)
package_manifest = json.loads((PACKAGE_DIR / "source_manifest.json").read_text(encoding="utf-8"))
if package_manifest["source_commit"] != SOURCE_COMMIT:
    raise RuntimeError("Το manifest έχει διαφορετική προέλευση.")
snapshot = package_manifest["runtime_snapshot"]
if (snapshot.get("complaint_primary_default") != "lexicon"
        or snapshot.get("lexicon_version") != EXPECTED_LEXICON_VERSION
        or snapshot.get("lexicon_sha256") != EXPECTED_LEXICON_SHA256):
    raise RuntimeError("Δεν είναι το πακέτο του συμφωνημένου local-primary source.")
PACKAGE_READY = True
print("Allowlist:", len(package_manifest["files"]), "αρχεία κώδικα + manifest.")


### 4. Λειτουργικός έλεγχος με πραγματικά βάρη

Το `check_triage_space.py` εκτελεί τις 16 επινοημένες/development περιπτώσεις με Jev off. Τα επτά απαιτούμενα regressions πρέπει να περάσουν. Σε κάθε αποτέλεσμα ελέγχει ότι το primary είναι το lexicon του manifest, το enhancement είναι disabled, δεν υπάρχει αποθήκευση/αίτημα Jev και ζητείται ανθρώπινος έλεγχος.

Δεν υπολογίζει precision/recall. Αν αποτύχει, η δημοσίευση παραλείπεται και το ZIP κρατά τη διάγνωση.

In [ ]:
SMOKE_READY = LIVE_READY = False
if not PACKAGE_READY:
    raise RuntimeError("Ολοκλήρωσε πρώτα το πακέτο.")
with (RUN_DIR / "smoke_execution.txt").open("w", encoding="utf-8") as log:
    smoke_process = subprocess.run([PYTHON, "-u", "scripts/check_triage_space.py",
        "--package", str(PACKAGE_DIR), "--output", str(SMOKE_REPORT)],
        cwd=REPO_DIR, env=base_env, stdout=log, stderr=subprocess.STDOUT, check=False)
smoke_report = json.loads(SMOKE_REPORT.read_text(encoding="utf-8")) if SMOKE_REPORT.exists() else None
SMOKE_READY = bool(smoke_process.returncode == 0 and smoke_report and smoke_report["passed"]
                       and smoke_report.get("regression_checks_passed") is True
                       and smoke_report.get("complaint_policy", {}).get("verified") is True
                       and smoke_report["complaint_policy"].get("primary") == "lexicon"
                       and smoke_report["complaint_policy"].get("lexicon_sha256") == EXPECTED_LEXICON_SHA256
                       and smoke_report["complaint_policy"].get("jev_enabled") is False)
print("Real-weight functional check:", "OK" if SMOKE_READY else "FAILED — δεν θα γίνει upload.")
if smoke_report:
    print("Cases:", len(smoke_report["records"]), "· προβλήματα εκτέλεσης:", smoke_report["problems"])
    print("Πραγματικό περιβάλλον:", smoke_report["runtime"])
else:
    print("Διάγνωση στο smoke_execution.txt του ZIP στο τέλος.")

if smoke_report:
    for row in smoke_report["records"]:
        if row.get("required_regression"):
            print(row["id"], "PASS" if row["regression_passed"] else "FAIL", "issues:", row["issues"], "actions:", row["actions"])


### 5. Ενημέρωση από τον λογαριασμό σου και έλεγχος του Space

Το token διαβάζεται αποκλειστικά από το Colab Secret. Πρώτα επικυρώνονται το πακέτο και το real-weight receipt· μετά αλλάζουν οι δύο δημόσιες Variables και ανεβαίνουν τα αρχεία σε ένα source commit με parent guard.

Ο publisher περιμένει το **νέο φορτωμένο snapshot**, έως 25 λεπτά. Επιβεβαιώνει το lexicon fingerprint, `Jev off`, ZeroGPU, τα επτά regressions και την τοπική πολιτική κάθε αποτελέσματος. Μια επιτυχής μεταφόρτωση με αποτυχημένο runtime διατηρείται ως τέτοια στο receipt· δεν θεωρείται επιτυχημένο demo.

In [ ]:
LIVE_READY = False
publication_code = None
if not SMOKE_READY:
    print("Upload skipped: ο λειτουργικός έλεγχος δεν πέρασε.")
else:
    publication_env = dict(base_env)
    if not DRY_RUN:
        from google.colab import userdata
        try:
            publication_env[SECRET_NAME] = userdata.get(SECRET_NAME)
        except Exception:
            raise RuntimeError("Χρειάζεται Colab Secret HF_TOKEN με write access και πρόσβαση από αυτό το notebook.") from None
    arguments = [PYTHON, "-u", "scripts/publish_triage_space.py", "--package", str(PACKAGE_DIR),
                 "--smoke-report", str(SMOKE_REPORT), "--output", str(PUBLICATION_REPORT), "--local-primary"]
    if DRY_RUN:
        arguments.append("--dry-run")
    if RESUME:
        arguments.append("--resume")
    try:
        with (RUN_DIR / "publish_execution.txt").open("w", encoding="utf-8") as log:
            completed = subprocess.run(arguments, cwd=REPO_DIR, env=publication_env,
                stdout=log, stderr=subprocess.STDOUT, check=False)
            publication_code = completed.returncode
    finally:
        secret_value = publication_env.pop(SECRET_NAME, None)
        log_path = RUN_DIR / "publish_execution.txt"
        if secret_value and log_path.is_file():
            content = log_path.read_text(encoding="utf-8")
            log_path.write_text(content.replace(secret_value, "[REDACTED]"), encoding="utf-8")
        del secret_value
    publication_report = json.loads(PUBLICATION_REPORT.read_text(encoding="utf-8")) if PUBLICATION_REPORT.exists() else None
    LIVE_READY = bool(publication_code == 0 and publication_report and publication_report["runtime_verified"] and publication_report.get("local_primary_verified") is True)
    print("Dry run:", DRY_RUN, "· exit code:", publication_code, "· live verified:", LIVE_READY)
    if publication_report:
        print("Space:", publication_report["url"])
        print("Commit:", publication_report["space_commit"])
        print("Upload verified:", publication_report["upload_verified"], "· runtime verified:", publication_report["runtime_verified"] and publication_report.get("local_primary_verified") is True)
    elif not DRY_RUN:
        print("Δεν επιβεβαιώθηκε upload. Δες publish_execution.txt στο ZIP.")


### 6. Κατέβασμα καταγραφής ακόμη και μετά από αποτυχία

Το ZIP περιλαμβάνει μόνο manifest και λειτουργικές καταγραφές αυτού του run. Αν ένα προηγούμενο κελί σταματήσει, τρέξε αυτό το κελί χειροκίνητα· δεν απαιτεί επιτυχία δημοσίευσης. Δεν συμπεριλαμβάνονται φάκελοι δεδομένων, CSV pilot, ετικέτες ή Secrets.

Καταγράφεται και το `setup_execution.txt`, ώστε μια αποτυχία εγκατάστασης/ελέγχου έκδοσης να μπορεί να διαγνωστεί.

In [ ]:
import zipfile
from google.colab import files

RUN_DIR.mkdir(parents=True, exist_ok=True)
archive = RUN_DIR / "triage_lexicon_space_update.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    if SETUP_LOG.is_file():
        bundle.write(SETUP_LOG, "setup_execution.txt")
    manifest_path = PACKAGE_DIR / "source_manifest.json"
    if manifest_path.is_file():
        bundle.write(manifest_path, "source_manifest.json")
    for name in ("smoke.json", "published.json", "smoke_execution.txt", "publish_execution.txt"):
        path = RUN_DIR / name
        if path.is_file():
            bundle.write(path, name)
print("Κατάσταση:", "experimental local-primary Space verified" if LIVE_READY else "publication not verified")
files.download(str(archive))

### 7. Μετά την εκτέλεση

Στείλε το `triage_lexicon_space_update.zip` για έλεγχο source/Hub/runtime πριν καταγραφεί ως δημοσιευμένη έκδοση. Αν δεν πέρασε, το ZIP δείχνει το τελευταίο επιβεβαιωμένο βήμα.

Το επόμενο ποιοτικό βήμα είναι το ανεξάρτητο whole-review pilot: [οδηγίες και offline commands](https://github.com/krimits/hotel-review-nlp/blob/02d6e8c7b7297bd87b4060b8f9f504421e60244c/docs/experiments/triage_lexicon_pilot/PROTOCOL.md). Θέλει πραγματικές, κατάλληλες κριτικές και ανθρώπινες ετικέτες. Δεν αντικαθίσταται από τα 500 primary-topic labels ή από επινοημένες κριτικές.